# Entrenar el tiny-LM de **nominaris** en Colab

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joelibaceta/nominaris/blob/main/tools/train/notebook.ipynb)

Entrena el modelo diminuto (GPT-2 ~12M params) y expórtalo a **ONNX int8** listo para el frontend. Ejecuta las celdas en orden.

**Necesitas subir 3 archivos** del repo cuando se te pida: `data/dataset.jsonl`, `tools/train/train.py`, `tools/train/export.py`.

> Sugerencia: en Colab, `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU` acelera el entrenamiento (también corre en CPU).

### 1. Dependencias

In [ ]:
!pip -q install "torch>=2.2" "transformers>=4.44" "tokenizers>=0.19" \
                "datasets>=2.20" "accelerate>=0.26" "optimum[onnxruntime]>=1.21" "onnx>=1.16"

### 2. Subir archivos (`dataset.jsonl`, `train.py`, `export.py`)

In [ ]:
import os
from google.colab import files
print('Elige: data/dataset.jsonl, train.py y export.py')
up = files.upload()
os.makedirs('data', exist_ok=True)
if 'dataset.jsonl' in up:
    os.replace('dataset.jsonl', 'data/dataset.jsonl')
assert os.path.exists('data/dataset.jsonl'), 'Falta data/dataset.jsonl'
assert os.path.exists('train.py') and os.path.exists('export.py'), 'Faltan train.py / export.py'
print('OK:', len(open('data/dataset.jsonl').read().splitlines()), 'lineas en el dataset')

### 3. Entrenar
Ajusta `--epochs` / `--n_embd` / `--n_layer` si quieres. Con GPU son minutos.

In [ ]:
!python train.py --data data/dataset.jsonl --out out --epochs 40 --n_embd 384 --n_layer 6 --dropout 0.1

### 4. Exportar a ONNX int8

In [ ]:
!python export.py --model out --out public/model
!ls -lh public/model/onnx/

### 5. Descargar el modelo
Descomprime en tu repo dentro de `public/` → queda `public/model/`. La app lo detecta y usa el tiny-LM automáticamente.

In [ ]:
!cd public && zip -r /content/model.zip model >/dev/null && echo OK
from google.colab import files
files.download('/content/model.zip')